# Лаборатория 2. Настоящий RAG: поиск по документам + ответ модели

**Что мы сделаем:** соберём работающего школьного бота, который отвечает на вопросы
по правилам школы — и честно говорит «в документах этого нет», когда ответа там нет.

Поиск будет настоящий (библиотека `rank_bm25`, тот же алгоритм, что в поисковых
системах), модель тоже настоящая. Всё вместе — примерно 40 строк.

**Что понадобится:** код класса от учителя.

In [ ]:
!pip -q install rank_bm25 openai

In [ ]:
import getpass
import os
from pprint import pprint

from openai import OpenAI

ADRES = "https://ai9.adelfos.ru/api/v1"
MODEL = "qwen/qwen3.7-flash"

try:
    from google.colab import userdata
    KOD_KLASSA = userdata.get("AI9_KOD") or os.environ.get("AI9_KOD")
except Exception:
    KOD_KLASSA = os.environ.get("AI9_KOD")

# Сервер проверит код, только когда мы обратимся к нему с ключом. Поэтому делаем один
# лёгкий запрос (список моделей) и, если код не принят, спрашиваем его заново.
client = None
while client is None:
    if not KOD_KLASSA:
        KOD_KLASSA = getpass.getpass("Код класса: ")
    client = OpenAI(base_url=ADRES, api_key=KOD_KLASSA)
    try:
        client.models.list()   # неверный код сервер не примет и ответит ошибкой
        print("Всё хорошо: код подошёл. Модель:", MODEL)
    except Exception:
        print("Код не подошёл — проверь его у учителя и введи заново.")
        client = None
        KOD_KLASSA = None      # после ошибки код из секретов и окружения больше не берём

## Шаг 1. Документы и нарезка на чанки

Вот «правила школы» — обычный текст, какой мог бы висеть на сайте. Первое, что делают
в любой такой системе, — режут документ на **чанки**, небольшие куски.

Почему не отдать модели весь текст сразу? Из темы 1: он может не поместиться
в контекстное окно, а середину длинного текста модель замечает хуже. Плюс каждый
лишний кусок — это лишние токены, то есть дороже и медленнее.

Режем по абзацам: в нашем тексте один абзац — одна тема. Это самый простой
разумный способ; бывают и хитрее, но начинать стоит с него.

In [ ]:
PRAVILA = """
Кружок робототехники работает по вторникам и четвергам. Начало в 15:40, кабинет 204.
Руководитель — Иванов Пётр Сергеевич. С собой нужна тетрадь, остальное выдают на месте.

Кружок рисования собирается по средам в 16:00 в кабинете 112. Краски и кисти каждый
приносит свои, бумагу выдаёт школа.

Столовая работает с 9:00 до 15:00. Завтрак с 9:00 до 9:30, обед с 12:20 до 13:00.
Горячее питание для учеников начальных классов бесплатное.

Библиотека открыта с 8:30 до 17:00 каждый день, кроме пятницы. Книги выдают на две
недели, учебники — на весь учебный год.

Школьный спортзал открыт до 19:00. После уроков там занимаются секции: волейбол
по понедельникам, баскетбол по четвергам.
"""

chanki = [kusok.strip().replace("\n", " ") for kusok in PRAVILA.strip().split("\n\n")]

print(f"Документ разрезан на {len(chanki)} чанков:\n")
for nomer, chank in enumerate(chanki):
    print(f"[{nomer}] {chank[:70]}...")

## Шаг 2. Настоящий поиск: BM25

`rank_bm25` — библиотека с алгоритмом **BM25**. Это не нейросеть, а честная математика,
которой пользуются поисковые системы уже лет тридцать. Идея у него умнее, чем «посчитать
совпадения слов»:

* слово, которое встречается почти во всех чанках (например, «работает»), весит мало —
  оно ничего не различает;
* редкое слово («библиотека») весит много — если оно совпало, это сильный сигнал;
* короткий чанк с совпадением ценится выше длинного: в длинном слово могло попасться случайно.

Именно поэтому BM25 не путает вопрос про библиотеку с текстом про столовую — а наш
самодельный поиск из урока путал.

Перед поиском текст надо разбить на слова (**токенизировать** — то же слово, что
в теме 1, только здесь кусочки это просто слова) и привести к одному виду:
нижний регистр, без знаков препинания.

In [ ]:
from rank_bm25 import BM25Okapi


def v_slova(text):
    """Текст -> список слов: нижний регистр, без знаков препинания."""
    ochishchennyy = "".join(bukva.lower() if bukva.isalnum() else " " for bukva in text)
    return ochishchennyy.split()


# Готовим поиск: он один раз изучает все чанки и запоминает, какое слово где встречается.
poisk = BM25Okapi([v_slova(chank) for chank in chanki])

vopros = "во сколько начинается кружок робототехники"
ocenki = poisk.get_scores(v_slova(vopros))

print(f"Вопрос: {vopros}\n")
print("Оценки всех чанков (чем больше, тем лучше подходит):")
for nomer, ocenka in enumerate(ocenki):
    print(f"  [{nomer}] {ocenka:5.2f}  {chanki[nomer][:55]}...")

Видно, что нужный чанк получил заметно больше остальных, а не просто «на один балл
больше». Это и есть разница между самоделкой и настоящим алгоритмом.

Теперь напишем функцию поиска: берём несколько лучших чанков, но только те, у которых
оценка больше нуля — то есть хоть что-то совпало.

In [ ]:
SKOLKO_BRAT = 2          # сколько чанков кладём в запрос модели


def nayti(vopros, skolko=SKOLKO_BRAT):
    """Возвращает список (оценка, чанк), лучшие сверху. Пустой список — ничего не нашлось."""
    ocenki = poisk.get_scores(v_slova(vopros))
    poryadok = sorted(range(len(chanki)), key=lambda i: ocenki[i], reverse=True)
    return [(ocenki[i], chanki[i]) for i in poryadok[:skolko] if ocenki[i] > 0]


for proba in ["когда работает библиотека", "во сколько баскетбол", "сколько стоит проезд в автобусе"]:
    naydennoe = nayti(proba)
    print(f"{proba!r}")
    if not naydennoe:
        print("   ничего не нашлось\n")
    for ocenka, chank in naydennoe:
        print(f"   {ocenka:5.2f}  {chank[:60]}...")
    print()

Посмотри на третий вопрос — про автобус. В правилах о нём нет ни слова, но поиск
всё равно что-то вернул: слова «сколько» и «в» встречаются и там, и там. Запомни это:
**поиск почти всегда что-нибудь находит**, даже когда находить нечего. Поэтому за ним
нужна вторая линия обороны — о ней шаг 3.

## Шаг 3. Собираем запрос к модели

Найденные чанки надо отдать модели вместе с вопросом — и обязательно с правилом
**отвечать только по этому тексту**. Без такого правила модель, не найдя ответа
в тексте, спокойно добавит его из головы: мы видели это в теме 1.

> **Грунтование** — требование отвечать только по выданному тексту и признаваться,
> когда ответа в нём нет.

Разберём запрос по частям:

* роль `system` — правила поведения, их пишет разработчик;
* роль `user` — найденные куски и сам вопрос;
* `temperature=0` — задача не творческая, ответ должен быть один и тот же.

In [ ]:
PRAVILO = (
    "Ты помощник школы. Отвечай на вопрос ТОЛЬКО по тексту из блока ДОКУМЕНТЫ.\n"
    "Если ответа в документах нет — ответь ровно: «В документах этого нет».\n"
    "Не добавляй ничего от себя. Отвечай одним-двумя предложениями."
)


def sprosit_bota(vopros):
    """Полный круг RAG: найти -> вложить в запрос -> ответить."""
    naydennoe = nayti(vopros)

    # Если поиск не нашёл вообще ничего, модель звать незачем: ответ известен заранее.
    if not naydennoe:
        return "В документах этого нет.", []

    dokumenty = "\n".join(f"- {chank}" for _, chank in naydennoe)
    soobshcheniya = [
        {"role": "system", "content": PRAVILO},
        {"role": "user", "content": f"ДОКУМЕНТЫ:\n{dokumenty}\n\nВОПРОС: {vopros}"},
    ]

    print("Что мы отправляем модели:")
    pprint(soobshcheniya, width=100, sort_dicts=False)

    otvet = client.chat.completions.create(
        model=MODEL,
        messages=soobshcheniya,
        temperature=0,
        max_tokens=200,
    )
    return otvet.choices[0].message.content.strip(), [chank for _, chank in naydennoe]

## Шаг 4. Проверяем бота на четырёх вопросах

Прежде чем запускать — угадай, на какие из них бот ответит правильно.

In [ ]:
VOPROSY = [
    "Во сколько начинается кружок робототехники?",
    "До скольких работает библиотека?",
    "Что нужно приносить с собой на рисование?",
    "Сколько стоит проезд в школьном автобусе?",
]

for vopros in VOPROSY:
    otvet, istochniki = sprosit_bota(vopros)
    print(f"❓ {vopros}")
    print(f"🤖 {otvet}")
    if istochniki:
        print(f"📄 что нашёл поиск: {istochniki[0][:60]}...")
    print()

## Шаг 5. Разбираем, что получилось — включая провал

Скорее всего, вышло так:

| Вопрос | Ответ | Что произошло |
|---|---|---|
| робототехника | верный | поиск нашёл нужное, модель ответила по тексту |
| библиотека | верный | то же самое |
| **рисование** | **«в документах этого нет»** | **поиск подсунул не тот чанк** |
| автобус | «в документах этого нет» | поиск принёс мусор, но модель не поддалась |

Два случая тут интересные, разберём оба.

**Провал на рисовании.** Ответ есть в документах: «Краски и кисти каждый приносит свои».
Но поиск его не нашёл. Почему — видно, если присмотреться к словам: в вопросе
«рисовани**е**», а в документе «рисовани**я**». Для BM25 это два **разных** слова,
он сравнивает их буква в букву.

Проверим догадку: зададим тот же вопрос словом в той же форме, что в документе.

In [ ]:
for proba in ["Что нужно приносить с собой на рисование?",
              "Что приносить на кружок рисования?"]:
    naydennoe = nayti(proba)
    nashlos = naydennoe[0][1][:55] if naydennoe else "ничего"
    print(f"{proba!r}\n   поиск нашёл: {nashlos}...\n")

Догадка подтвердилась: достаточно совпадения словоформы — и нужный чанк находится.

Это не мелкая придирка, а главная слабость поиска по словам. Лечат её двумя способами:
приводят слова к общей основе (грубо — обрезают окончания) либо ищут по смыслу.
Вторым займёмся в теме 3.

**Спасение на автобусе.** Здесь поиск, наоборот, ошибся в другую сторону: принёс чанк
про кружок рисования, потому что совпали слова «сколько» и «в». Модель получила текст,
не имеющий отношения к вопросу, — и всё равно ответила правильно: «в документах этого нет».

Сработало **грунтование**. Сравни с темой 1, где та же самая модель уверенно сочиняла
сюжет несуществующей книги. Изменилась не модель, а устройство системы.

Вывод, ради которого написана эта лаборатория:

> В RAG две части, и ломаются они по-разному. **Поиск** отвечает за то, найдётся ли
> нужный кусок. **Грунтование** — за то, что модель не выдумает, если кусок не тот.
> Нужны обе.

И ещё: заметил ли ты провал на рисовании сам, до того как прочитал разбор? Вот почему
в теме 5 мы научимся измерять качество числом, а не глазами.

## Шаг 6. Где ещё эта конструкция ломается

Ещё один случай, который стоит увидеть своими глазами.

In [ ]:
for vopros in ["Когда занятия по волейболу?", "Где можно поесть в школе?"]:
    naydennoe = nayti(vopros)
    print(f"❓ {vopros}")
    print(f"   поиск нашёл: {naydennoe[0][1][:60] if naydennoe else 'ничего'}...")
    otvet, _ = sprosit_bota(vopros)
    print(f"🤖 {otvet}\n")

Про волейбол бот ответит, а вот «где можно поесть» почти наверняка провалит: в правилах
написано «столовая», а слова «поесть» там нет вовсе. Ни одного общего слова — BM25 бессилен.

Человек видит, что «поесть» и «столовая» про одно и то же. Компьютер, сравнивающий буквы,
не видит. Чтобы он увидел, нужен поиск по смыслу — это следующая тема.

## Попробуй сам

1. Замени `PRAVILA` на свои документы: правила твоей секции, конспект по предмету,
   описание игры. Работает ли бот на них?
2. Задай вопрос, ответ на который разбросан по двум чанкам сразу (например,
   «какие кружки бывают и когда»). Поставь `SKOLKO_BRAT = 3` — стало лучше?
3. Убери из `PRAVILO` строчку про «В документах этого нет» и задай вопрос про автобус
   ещё раз, закомментировав ранний выход в `sprosit_bota`. Что ответит модель?
4. Спроси одно и то же дважды при `temperature=0` — ответы совпали? А при `temperature=1.5`?

## Что унести с собой

* **RAG** = сначала найти нужные куски, потом ответить по ним. Модель не «знает»
  твоих документов — она их читает прямо в запросе.
* **Чанк** — кусок документа. Режем, потому что целиком не поместится и середину
  модель замечает хуже.
* **BM25** — настоящий алгоритм поиска по словам: редкие слова весят больше частых,
  короткие чанки ценятся выше длинных.
* **Грунтование** — правило «отвечай только по тексту, иначе скажи, что не знаешь».
  Именно оно превращает выдумщика из темы 1 в надёжного помощника.
* Если поиск не нашёл ничего — не зови модель вовсе. Нет шанса выдумать — нет выдумки.
* Показывай источник ответа: непроверяемый ответ ИИ стоит немного.
* Поиск по словам не понимает синонимов — это чинится в теме 3.